# The State of the Antarct Ice Shelves Data Linkages
This interactive notebook enables you to investigate data linkages between key variables across various ice shelves.

Key features:
- Feature 1
- Feature 2
- Feature 3

In [1]:
%load_ext autoreload
%autoreload 2

import ipywidgets as widgets
from traitlets import traitlets
from dtc_query_client import Configuration, ApiClient, GenericApi, StateAndFateApi

First we need to authenticate to interact with the DTC Ice Sheets system. To do this please navigate to the following [webpage](https://query.dtc-ice-sheets.org/auth/get-token) to generate an API token and then enter it below where requested:

In [37]:
def on_button_clicked(b):
    with credentials_output:
        credentials_output.clear_output()
        if credentials_box.value == "":
            print("Please enter an API token before submitting.")
        else:
            print(f"API token submitted: {credentials_box.value}")

            client = authenticate_with_token(credentials_box.value)

    b.value = client

def authenticate_with_token(token):
    config = Configuration(
        host="https://query.dtc-ice-sheets.org",
        access_token=token
    )
    client = ApiClient(config)
    return client

class LoadedButton(widgets.Button):
    """A button that can hold a value as an attribute."""

    def __init__(self, value=None, *args, **kwargs):
        super(LoadedButton, self).__init__(*args, **kwargs)
        # Create the value attribute.
        self.add_traits(value=traitlets.Any(value))

def update_client(b):
    return b.value

async def get_ice_shelves(client):
    return await StateAndFateApi(client).list_ice_shelves()

async def get_datasets(client):
    return await GenericApi(client).dataset_overviews()

async def ice_shelf_selector(client):
    
    ice_shelves = await get_ice_shelves(client=client)

    return widgets.Dropdown(
        options=ice_shelves,
        value=ice_shelves[0],
        description='Ice Shelf:',
        disabled=False,
    )

async def dataset_selector(client):

    datasets = await get_datasets(client=client)

    datasets = [dataset.dataset_id for dataset in datasets if dataset.dataset_id != ""]

    return widgets.SelectMultiple(
        options=datasets,
        value=[datasets[0]],
        rows=10,
        description='Datasets',
        disabled=False,
        layout=widgets.Layout(width='80%')
    )

def analysis_type_selector():

    return widgets.ToggleButtons(options=['Correlation', 'Cross-Correlation', 'Granger Causality'],
        description='Select Analysis Type:',
        disabled=False,
        button_style='info', # 'success', 'info', 'warning', 'danger' or ''
    )

In [13]:
credentials_box = widgets.Text(
    value='',
    placeholder='Enter your API token here...',
    description='API token:',
    disabled=False
)

credentials_button = LoadedButton(
    description='submit API token',
    disabled=False,
    button_style='info', # 'success', 'info', 'warning', 'danger' or ''
    tooltip='',
    icon=''
)

credentials_button.on_click(on_button_clicked)

credentials_output = widgets.Output()

credentials_container = [credentials_box, credentials_button, credentials_output]

display(widgets.Box(credentials_container))

Box(children=(Text(value='', description='API token:', placeholder='Enter your API token here...'), LoadedButt…

In [4]:
client = credentials_button.value

Please choose the ice shelf:

In [5]:
ice_shelf_selection = await ice_shelf_selector(client)
display(ice_shelf_selection)

Dropdown(description='Ice Shelf:', options=('abbot', 'ainsworth', 'alison', 'amery', 'andreyev', 'astrolabe', …

Please select the datasets you would like to include in the analysis:

In [38]:
dataset_selection = await dataset_selector(client)
display(dataset_selection)

SelectMultiple(description='Datasets', index=(0,), layout=Layout(width='80%'), options=('CLIMATE_DT_SCENARIO_M…

Select an ice shelf:

In [36]:
analysistype_selection = analysis_type_selector()
display(analysistype_selection)

ToggleButtons(button_style='info', description='Select Analysis Type:', options=('Correlation', 'Cross-Correla…